# Vision-Language Models — The ViT-MLP-LLM Pattern Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: The projector

The part you will train most often. 2-4 layer MLP with GELU.

In [ ]:
```python

import torch

import torch.nn as nn

class Projector(nn.Module):

    def __init__(self, vit_dim=768, llm_dim=4096, hidden=4096):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(vit_dim, hidden),

            nn.GELU(),

            nn.Linear(hidden, llm_dim),

        )

    def forward(self, x):

        return self.net(x)

In [ ]:
```

Input is a `(N_patches, d_vit)` token tensor. Output is `(N_patches, d_llm)`. The LLM treats every output row as just another token.

### Step 2: Assemble ViT-MLP-LLM end-to-end

Skeleton of the forward pass for a minimal VLM. Real code uses `transformers`; this is the conceptual layout.

In [ ]:
```python

class MinimalVLM(nn.Module):

    def __init__(self, vit, projector, llm, image_token_id):

        super().__init__()

        self.vit = vit

        self.projector = projector

        self.llm = llm

        self.image_token_id = image_token_id  # placeholder token in text prompt

    def forward(self, image, input_ids, attention_mask):

        # 1. vision features

        vision_tokens = self.vit(image)                     # (B, N_patches, d_vit)

        vision_embeds = self.projector(vision_tokens)       # (B, N_patches, d_llm)

        # 2. text embeddings

        text_embeds = self.llm.get_input_embeddings()(input_ids)  # (B, M, d_llm)

        # 3. replace image placeholder tokens with vision embeds

        merged = self._merge(text_embeds, vision_embeds, input_ids)

        # 4. run LLM

        return self.llm(inputs_embeds=merged, attention_mask=attention_mask)

    def _merge(self, text_embeds, vision_embeds, input_ids):

        out = text_embeds.clone()

        expected = vision_embeds.size(1)

        for b in range(input_ids.size(0)):

            positions = (input_ids[b] == self.image_token_id).nonzero(as_tuple=True)[0]

            if len(positions) != expected:

                raise ValueError(

                    f"batch item {b} has {len(positions)} image tokens but vision_embeds has {expected} patches."

                    " Every sample in the batch must be pre-padded to the same number of image placeholder tokens.")

            out[b, positions] = vision_embeds[b]

        return out

In [ ]:
```

The `<image>` placeholder token in the text gets replaced with real image embeddings — same pattern LLaVA, Qwen-VL, and InternVL use.

### Step 3: CMER computation

A lightweight runtime check.

In [ ]:
```python

import torch.nn.functional as F

def cross_modal_error_rate(image_emb, text_emb, text_confidence, sim_threshold=0.25, conf_threshold=0.8):

    """

    image_emb, text_emb: embeddings of image and generated text (normalised internally)

    text_confidence:     mean per-token probability in [0, 1]

    Returns:             fraction of high-confidence outputs with low image-text alignment

    """

    image_emb = F.normalize(image_emb, dim=-1)

    text_emb = F.normalize(text_emb, dim=-1)

    sim = (image_emb * text_emb).sum(dim=-1)        # cosine similarity

    high_conf_low_sim = (text_confidence > conf_threshold) & (sim < sim_threshold)

    return high_conf_low_sim.float().mean().item()

In [ ]:
```

Treat CMER as a production KPI. Monitor it per endpoint, per prompt type, per customer. Rising CMER indicates the model is starting to hallucinate on some input distribution.

### Step 4: Toy VLM classifier (runnable)

Demonstrate the projector trains. Fake "ViT features" go in; a tiny LLM-style token predicts a class.

In [ ]:
```python

class ToyVLM(nn.Module):

    def __init__(self, vit_dim=32, llm_dim=64, num_classes=5):

        super().__init__()

        self.projector = Projector(vit_dim, llm_dim, hidden=64)

        self.head = nn.Linear(llm_dim, num_classes)

    def forward(self, vision_tokens):

        projected = self.projector(vision_tokens)

        pooled = projected.mean(dim=1)

        return self.head(pooled)

In [ ]:
```

One can fit this on synthetic (feature, class) pairs in under 200 steps — enough to show the projector pattern works.

## Exercises

In [ ]:
1. **(Easy)** Run three prompts ("what is this?", "count the objects", "describe the scene") through any open VLM on five images. Score each answer as correct / partially correct / hallucinated by hand. Compute a first-pass CMER-like rate.
2. **(Medium)** Fine-tune Qwen2.5-VL-3B or LLaVA-1.6-7B with LoRA (rank 16) on 500 images of a target domain with captions. Compare zero-shot vs fine-tuned MMBench-style accuracy.
3. **(Hard)** Replace the VLM's image encoder with DINOv3 instead of its default SigLIP/CLIP. Re-train only the projector (frozen LLM + frozen DINOv3). Measure whether dense-prediction tasks (counting, spatial reasoning) improve.